# Adding New Data (Append w/ Threshold)
Threshold = 0.4

In [1]:
import pandas as pd
from numpy import dot
from numpy.linalg import norm

In [2]:
def get_cosine(embedding_1, embedding_2):
    cos_sim = dot(embedding_1, embedding_2)/(norm(embedding_1)*norm(embedding_2))
    return cos_sim

def append_to_sense_inventory(new_inventory, old_inventory, threshold):
    counter = 0
    combined = old_inventory

    # add new senses
    for new_index, new_row in new_inventory.iterrows():
        high_sim = 0
        temp = 0

        #print(f"\nnew inventory {new_row['sense_id']}...")

        # time efficient slice
        temp_df = old_inventory.loc[old_inventory['word'] == new_row['word']]

        for old_index, old_row in temp_df.iterrows():
            if new_row['word'] == old_row['word']:
                temp = get_cosine(new_row['sense_embedding'], old_row['sense_embedding'])
                #print(f"old inventory {old_row['sense_id']} & new inventory {new_row['sense_id']} sim: {temp}")

            if temp >= high_sim:
                high_sim = temp
        # add sense if it has 'threshold' or lower as its highest similarity among the senses for that word
        #print(f"highest cosine similarity for {new_row['sense_id']} is {str(high_sim)}!")
        if high_sim <= threshold:
            print(f"ADDED: {new_row}")
            combined = combined.append(new_row, ignore_index=True)
            counter += 1

    # sort senses
    combined = combined.sort_values('word')

    # rename all sense id
    word = new_inventory.iloc[0]['word']
    id = 0

    for index, row in combined.iterrows():
        if word == row['word']:
            combined.loc[index, 'sense_id'] = word+'_'+str(id)
            id += 1
        else:
            word = row['word']
            id = 0
            combined.loc[index, 'sense_id'] = word+'_'+str(id)
            id += 1

    # reset index
    combined = combined.reset_index(drop=True)

    print(f"\n# of new senses is {str(counter)}!")
    return combined


## Load old and new sense inventories

In [3]:
old_inventory = pd.read_pickle("old_sense_inventory.pkl")
old_inventory

,word,sense_id,sense_embedding,example_sentences,contextual_info,pos
0,matino,matino_0,"[0.0039200424, 0.03125514, 0.044149794, -0.006...","[halatang wala ka ng gagawing matino HAHAHA, t...",{'twitter': {'2021': 10}},FW
1,matino,matino_1,"[0.014258224, -0.007326535, 0.04272052, -0.020...",[May mga magulang na matino ang isip at katawa...,"{'google_books': {'2018': 2}, 'twitter': {'202...",FW
2,matino,matino_2,"[0.0058104345, -0.0023813697, 0.04274508, -0.0...","[ng pelikula. Maganda at matino., kasama sa pe...","{'bandera': {'2014': 2, '2018': 2, '2021': 1, ...",VB
3,matino,matino_3,"[-0.0070238346, 0.014438677, 0.029195312, -0.0...","[akong sinabing matino., Duda ako sa matino!, ...","{'twitter': {'2021': 7}, 'bandera': {'2020': 1...",NN
4,maayos,maayos_0,"[0.010224508, 0.022195492, 0.058941443, -0.019...","[buti nalang nakasagot naman ng maayos., sana ...",{'twitter': {'2021': 10}},JJ
...,...,...,...,...,...,...
7078,tuod,tuod_1,"[-0.0010470327, 0.024433836, 0.05405687, -0.00...","[sabihin ay parang tuod., Ano ko tuod walang p...","{'balita': {'2017': 2}, 'twitter': {'2021': 6}...",FW
7079,bumaling,bumaling_0,"[-0.002633381, 0.05177117, 0.050690066, 0.0117...",[tingin sakin tas nung bumaling kay joms ay ng...,"{'twitter': {'2021': 7}, 'google_books': {'202...",VB
7080,bumaling,bumaling_1,"[0.029594356, 0.004925939, 0.008006528, 0.0289...","[puso ay ganap nang bumaling sa Diyos, ipinaki...","{'google_books': {'2019': 6, '2021': 4}}",NN
7081,anas,anas_0,"[-0.033860113, 0.0042065405, 0.05422906, 0.003...","[Ni graduate nadaw kog laag anas mama HAHAAHA,...",{'twitter': {'2021': 10}},NN


In [4]:
new_inventory = pd.read_pickle("new_sense_inventory.pkl")
new_inventory

,word,sense_id,sense_embedding,example_sentences,contextual_info,pos
0,matino,matino_0,"[0.008358148, 0.034379065, 0.049863774, -0.010...","[pa nga kong nagagawang matino chudey huhu, ak...",{'twitter': {'2021': 10}},FW
1,matino,matino_1,"[-0.0004953388, 0.030745406, 0.01920554, -0.04...","[ng matino., kung hindi maayos at matino., Iil...","{'gma': {'2014': 1}, 'twitter': {'2021': 7}, '...",NN
2,matino,matino_2,"[-0.0037480977, 0.005228241, 0.04773019, -0.02...","[kala mo naman talaga matino, paligaw at ako y...",{'twitter': {'2021': 10}},FW
3,matino,matino_3,"[0.0103537, -0.014296556, 0.04497205, -0.02459...","[asawa ko, at lubhang matino at palaging magal...","{'google_books': {'2021': 1, '2018': 1}, 'bali...",NN
4,maayos,maayos_0,"[-0.0037643071, 0.019667514, 0.046444785, -0.0...","[nakahinga ako ng maayos, ket konti, ngyon edi...",{'twitter': {'2021': 10}},JJ
...,...,...,...,...,...,...
5721,tuod,tuod_1,"[-0.01778888, -0.038977534, 0.044225916, 0.019...","[ko lang gid no sa tuod lang, nang magbalik gu...",{'twitter': {'2021': 10}},NN
5722,bumaling,bumaling_0,"[0.0004811151, 0.033824384, 0.038164582, 0.000...",[naman sila sa pagbabangayan tsaka bumaling sa...,"{'twitter': {'2021': 3}, 'balita': {'2015': 1,...",VB
5723,bumaling,bumaling_1,"[0.034572523, 0.014924703, 0.0027387284, 0.030...","[puso na ganap na bumaling sa Kanya, ito ang u...","{'google_books': {'2019': 2, '2021': 3}}",NN
5724,anas,anas_0,"[-0.016895104, 0.0038351163, 0.059679933, 0.00...",[bag ong victim daw niya anas chingching hahah...,{'twitter': {'2021': 5}},FW


In [5]:
combined = append_to_sense_inventory(new_inventory=new_inventory,
                                           old_inventory=old_inventory,threshold=0.4) # 40% and below threshold
combined

ADDED: word                                                           trabaho
sense_id                                                     trabaho_2
sense_embedding      [0.0015942536, -0.045249186, -0.013914352, -0....
example_sentences    [kinalaman ang isyu sa trabaho o desisyon ng S...
contextual_info      {'gma': {'2012': 2}, 'balita': {'2017': 1, '20...
pos                                                                 NN
Name: 12, dtype: object
ADDED: word                                                           matatag
sense_id                                                     matatag_0
sense_embedding      [0.015831824, 0.024847796, -0.022892734, -0.01...
example_sentences    [sapat, maaasahan at matatag na power supply a...
contextual_info      {'balita': {'2016': 2, '2015': 1}, 'radyoinqui...
pos                                                                 JJ
Name: 23, dtype: object
ADDED: word                                                           matanda
sense_id

ADDED: word                                                      samantalahin
sense_id                                                samantalahin_1
sense_embedding      [0.009562215, -0.02592529, -0.023388721, -0.01...
example_sentences    [Reyes na dapat samantalahin ng mga koponan an...
contextual_info      {'balita': {'2018': 2, '2016': 1, '2017': 1}, ...
pos                                                                 VB
Name: 731, dtype: object
ADDED: word                                                           gumamit
sense_id                                                     gumamit_2
sense_embedding      [0.084563605, 0.007306882, -0.008592385, -0.01...
example_sentences    [) ang mga motorista na gumamit ng mga alterna...
contextual_info      {'balita': {'2018': 1, '2015': 2, '2016': 2, '...
pos                                                                 VB
Name: 735, dtype: object
ADDED: word                                                       pangasiwaan
sense_

ADDED: word                                                             siyam
sense_id                                                       siyam_2
sense_embedding      [-0.038666308, -0.047888495, 0.031757634, 0.03...
example_sentences    [ng lindol ay siyam na kilometro sa kanluran n...
contextual_info      {'bandera': {'2019': 1, '2020': 1}, 'radyoinqu...
pos                                                                 JJ
Name: 1467, dtype: object
ADDED: word                                                             siyam
sense_id                                                       siyam_3
sense_embedding      [0.017925331, 0.0019772165, -0.01171122, 0.036...
example_sentences    [Nakuha sa kanila ang siyam na sachet ng hinih...
contextual_info      {'gma': {'2018': 2, '2021': 1}, 'bandera': {'2...
pos                                                                 JJ
Name: 1468, dtype: object
ADDED: word                                                              anim
sens

ADDED: word                                                              hita
sense_id                                                        hita_1
sense_embedding      [0.038610198, -0.026035508, 0.010592712, 0.008...
example_sentences    [iniindang pamamaga sa kanang hita, sa naisala...
contextual_info          {'balita': {'2019': 2, '2017': 2, '2016': 1}}
pos                                                                 NN
Name: 2576, dtype: object
ADDED: word                                                             bibig
sense_id                                                       bibig_2
sense_embedding      [-0.02053077, -0.037800785, 0.03784222, 0.0106...
example_sentences    [Tani gihipos ko na lang bibig ko, budlay na g...
contextual_info                               {'twitter': {'2021': 5}}
pos                                                                 NN
Name: 2588, dtype: object
ADDED: word                                                              test
sens

ADDED: word                                                             buwan
sense_id                                                       buwan_3
sense_embedding      [0.015549396, -0.03750023, -0.031096313, 0.011...
example_sentences    [sa finals ng kompetisyon sa buwan ng Abril, s...
contextual_info      {'radyoinquirer': {'2018': 2}, 'balita': {'201...
pos                                                                 NN
Name: 3816, dtype: object
ADDED: word                                                              bula
sense_id                                                        bula_3
sense_embedding      [0.01297381, 0.057879508, 0.012686171, -0.0693...
example_sentences    [210909 bula bath ni Para mawala yung kaba, 21...
contextual_info                               {'twitter': {'2021': 5}}
pos                                                                 NN
Name: 3895, dtype: object
ADDED: word                                                          ika-anim
sens

,word,sense_id,sense_embedding,example_sentences,contextual_info,pos
0,abala,abala_0,"[0.011410227, 0.005817835, -0.008558039, 0.030...","[ng gabi, habang abala umano sa pagtitinda ang...","{'balita': {'2016': 2, '2020': 1, '2018': 1}, ...",NN
1,abala,abala_1,"[0.035149045, 0.062080346, 0.018494673, -0.062...","[pasensya na sa abala!, sa abala., po, pasensy...","{'twitter': {'2021': 8}, 'radyoinquirer': {'20...",NN
2,abala,abala_2,"[0.018175527, 0.015186829, 0.019982383, 0.0261...",[sa piitan nitong Miyerkules habang abala ang ...,"{'gma': {'2014': 1, '2017': 1}, 'radyoinquirer...",NN
3,abala,abala_3,"[0.0139153, 0.017601732, 0.018305968, -0.04806...","[ito sa abala niyang schedule., noon ay naging...","{'balita': {'2016': 1, '2014': 1, '2019': 2, '...",JJ
4,abalahin,abalahin_0,"[-0.0037691495, 0.033807106, 0.021705411, -0.0...","[Huwag mo na siyang abalahin., Huwag mong abal...","{'twitter': {'2021': 3}, 'balita': {'2015': 1,...",VB
...,...,...,...,...,...,...
7172,zombie,zombie_1,"[0.023807092, 0.043104503, 0.043869168, -0.041...",[: Hala magiging zombie kanaKuya ko : okay lan...,{'twitter': {'2021': 6}},NN
7173,zombie,zombie_2,"[-0.026886176, -0.020745393, 0.047452476, -0.0...","[sa mga silibgan nga maging zombie daw., nalan...",{'twitter': {'2021': 10}},NN
7174,zombie,zombie_3,"[0.013196042, 0.013754795, 0.038955215, -0.020...","[at hindi dahil sa zombie hahahahaha, kala mo ...",{'twitter': {'2021': 10}},NN
7175,zoology,zoology_0,"[0.008698043, 0.00089879415, 0.058598656, -0.0...","[ako gumawa ng assignment sa zoology huhu, nab...",{'twitter': {'2021': 10}},NN


In [6]:
combined

,word,sense_id,sense_embedding,example_sentences,contextual_info,pos
0,abala,abala_0,"[0.011410227, 0.005817835, -0.008558039, 0.030...","[ng gabi, habang abala umano sa pagtitinda ang...","{'balita': {'2016': 2, '2020': 1, '2018': 1}, ...",NN
1,abala,abala_1,"[0.035149045, 0.062080346, 0.018494673, -0.062...","[pasensya na sa abala!, sa abala., po, pasensy...","{'twitter': {'2021': 8}, 'radyoinquirer': {'20...",NN
2,abala,abala_2,"[0.018175527, 0.015186829, 0.019982383, 0.0261...",[sa piitan nitong Miyerkules habang abala ang ...,"{'gma': {'2014': 1, '2017': 1}, 'radyoinquirer...",NN
3,abala,abala_3,"[0.0139153, 0.017601732, 0.018305968, -0.04806...","[ito sa abala niyang schedule., noon ay naging...","{'balita': {'2016': 1, '2014': 1, '2019': 2, '...",JJ
4,abalahin,abalahin_0,"[-0.0037691495, 0.033807106, 0.021705411, -0.0...","[Huwag mo na siyang abalahin., Huwag mong abal...","{'twitter': {'2021': 3}, 'balita': {'2015': 1,...",VB
...,...,...,...,...,...,...
7172,zombie,zombie_1,"[0.023807092, 0.043104503, 0.043869168, -0.041...",[: Hala magiging zombie kanaKuya ko : okay lan...,{'twitter': {'2021': 6}},NN
7173,zombie,zombie_2,"[-0.026886176, -0.020745393, 0.047452476, -0.0...","[sa mga silibgan nga maging zombie daw., nalan...",{'twitter': {'2021': 10}},NN
7174,zombie,zombie_3,"[0.013196042, 0.013754795, 0.038955215, -0.020...","[at hindi dahil sa zombie hahahahaha, kala mo ...",{'twitter': {'2021': 10}},NN
7175,zoology,zoology_0,"[0.008698043, 0.00089879415, 0.058598656, -0.0...","[ako gumawa ng assignment sa zoology huhu, nab...",{'twitter': {'2021': 10}},NN


In [8]:
combined.to_csv("updated_sense_inventory.csv", index=False)
combined.to_pickle("updated_sense_inventory.pkl", protocol=3)